
# <font color="green">Allocation-Intensive Programs (Binary Search Tree)</font>

- Explore a more realistic workload involving heavy memory allocation.
- Use the binary search tree program (`bst`) introduced in the `recursive_type` topic.
- The program first builds a tree of $M$ elements, then repeats $N$ times: insert one element and remove one, keeping the tree size constant at $M$.

## Work

- Compile (if necessary) and run each program.
- Suggested parameters: $M = 100000$ and $N = 500000$.
- Adjust GC knobs to reduce the reported time per allocation.





# 1. AI tutor

In [ ]:
import heytutor

## 1-1. Basics
```
%%hey [problem_file=...] [answer_file=...]
```

### 1-1-1. Builtin variables
* `{file:FILENAME}` is the content of FILE
* `{files:PATTERN}` is the content of files matching PATTERN (e.g., `{files:go/bst/*/*.go}`)
* `{bash[-1]}` is the output of the last `%%bash_` cell, `{bash[-2]}` that of the second last `%%bash_` cell, etc.
* `{problem}` is the content of the file you specified by `%%hey problem_file=foo.md`
* `{answer}` is the content of the file you specified by `%%hey answer_file=go/foo.go`


# 2. C/C++ for comparison

## 2-1. AI tutor

In [ ]:
import heytutor

## 2-2. Set up a module

In [ ]:
%%bash_

mkdir -p cc/bst

## 2-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* If you edit the file outside Jupyter, <font color=red>be careful not to overwrite it with an empty file</font>

In [ ]:
%%writefile_ cc/bst/bst.cc


#include <assert.h>
#include <err.h>
#include <stdint.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <utility>

int64_t time_ns() {
  struct timespec ts[1];
  if (clock_gettime(CLOCK_REALTIME, ts) == -1) err(1, "clock_gettime");
  return ts->tv_nsec + ts->tv_sec * 1000L * 1000L * 1000L;
}

// random number generator state
struct RandState {
  uint64_t x;
  uint64_t next();
  RandState(uint64_t x_) { x = x_; }
};

// next number
uint64_t RandState::next() {
  uint64_t a = 0x5DEECE66D;
  uint64_t c = 0xB;
  uint64_t m = (1UL << 48) - 1;
  x = (a * x + c) & m;
  return x >> 17;
}

// binary search tree
struct BinSearchTree {
  uint64_t val;
  uint64_t lc;
  BinSearchTree * left;
  uint64_t rc;
  BinSearchTree * right;
  BinSearchTree(uint64_t val, uint64_t lc, BinSearchTree * left, uint64_t rc, BinSearchTree * right) {
    this->val = val;
    this->lc = lc;
    this->left = left;
    this->rc = rc;
    this->right = right;
  }
};

// insert x to a tree
BinSearchTree * insert(BinSearchTree * t, uint64_t x) {
  if (!t) {
    return new BinSearchTree(x, 0, nullptr, 0, nullptr);
  } else {
    auto val = t->val;
    auto lc = t->lc;
    auto left = t->left;
    auto rc = t->rc;
    auto right = t->right;
    delete t;
    if (x <= val) {
      return new BinSearchTree(val, lc + 1, insert(left, x), rc, right);
    } else {
      return new BinSearchTree(val, lc, left, rc + 1, insert(right, x));
    }
  }
}

// remove the nth element in the tree (0-based), and return the removed value and the new tree
std::pair<uint64_t, BinSearchTree *> remove_nth(BinSearchTree * t, uint64_t n) {
  if (!t) {
    err(1, "remove_nth : empty tree");
  } else if (n < t->lc) {
    auto [val, left_] = remove_nth(t->left, n);
    auto val_ = t->val;
    auto lc_ = t->lc - 1;
    auto rc_ = t->rc;
    auto right_ = t->right;
    delete t;
    return {val, new BinSearchTree(val_, lc_, left_, rc_, right_)};
  } else if (n == t->lc) {
    if (t->lc < t->rc) {
      auto [val, right_] = remove_nth(t->right, 0);
      auto val_ = t->val;
      auto lc_ = t->lc;
      auto left_ = t->left;
      auto rc_ = t->rc - 1;
      delete t;
      return {val_, new BinSearchTree(val_, lc_, left_, rc_, right_)};
    } else if (t->left) {
      auto [val, left_] = remove_nth(t->left, t->lc - 1);
      auto val_ = t->val;
      auto lc_ = t->lc - 1;
      auto rc_ = t->rc;
      auto right_ = t->right;
      delete t;
      return {val_, new BinSearchTree(val_, lc_, left_, rc_, right_)};
    } else {
      auto val_ = t->val;
      delete t;
      return {val_, nullptr};
    }
  } else {
    auto [val, right_] = remove_nth(t->right, n - t->lc - 1);
    auto val_ = t->val;
    auto lc_ = t->lc;
    auto left_ = t->left;
    auto rc_ = t->rc - 1;
    delete t;
    return {val, new BinSearchTree(val_, lc_, left_, rc_, right_)};
  }
}

void dump(BinSearchTree * t, uint64_t n) {
  if (t && n > 0) {
    dump(t->left, n);
    if (t->lc < n) {
      printf("%lu ", t->val);
      if (t->lc + 1 < n) {
	dump(t->right, n - t->lc - 1);
      }
    }
  }
}

int main(int argc, char ** argv) {
  printf("lang = cplus\n");
  int64_t m = (argc > 1 ? atol(argv[1]) : 100L * 1000L);
  int64_t n = (argc > 2 ? atol(argv[2]) : m / 2);
  RandState rg(123456);
  printf("make a tree of m = %ld nodes\n", m);
  BinSearchTree * t = nullptr;
  for (int64_t i = 0; i < m; i++) {
    uint64_t x = rg.next() % 1000000000;
    t = insert(t, x);
  }
  printf("insert/delete n = %ld times\n", n);
  auto t0 = time_ns();
  for (int64_t i = 0; i < n; i++) {
    uint64_t x = rg.next() % 1000000000;
    uint64_t k = rg.next() % (m + 1);
    t = insert(t, x);
    auto vt = remove_nth(t, k);
    t = vt.second;
  }
  auto t1 = time_ns();
  printf("dump the first 5 elements in the tree : ");
  dump(t, 5);
  printf("\n");
  printf("took %ld nsec to insert/remove %ld elements (%f nsec/elem)\n", (t1 - t0), n, (t1 - t0) / (double)n);
  return 0;
}



In [ ]:
%%bash_

cat cc/bst/bst.cc

## 2-4. Build

In [ ]:
%%bash_

cd cc/bst
g++ -o bst -Wall -Wextra -O3 bst.cc

## 2-5. Run

In [ ]:
%%bash_

lang=cc
exe=cc/bst/bst

M=100000
N=500000

/usr/bin/time ${exe} ${M} ${N}


## 2-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=bst.md

Problem:
{problem}

...

# 3. Go

## 3-1. AI tutor

In [ ]:
import heytutor

## 3-2. Set up a module

In [ ]:
%%bash_

export PATH=${PATH}:~/.local/go/bin:~/go/bin
mkdir -p go/bst
cd go/bst
go mod init bst

* Note: when you run `go` or other Go commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/go/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 3-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* If you edit the file outside Jupyter, <font color=red>be careful not to overwrite it with an empty file</font>

In [ ]:
%%writefile_ go/bst/bst.go

package main
import "fmt"
import "os"
import "strconv"
import "github.com/loov/hrtime"

// ---------- utility ----------

func time_ns() int64 {
	return int64(hrtime.Now())
}

// ---------- random number generator state ----------
type RandState struct {
    x  uint64
}

// next number
func (rg * RandState) next() uint64 {
	x := rg.x
	var a uint64 = 0x5DEECE66D
	var c uint64 = 0xB
	var m uint64 = (uint64(1) << 48) - 1
	rg.x = (a * x + c) & m
	return rg.x >> 17
}

// ---------- binary search tree ----------
type BinSearchTree struct {
	val uint64
	lc uint64
	left * BinSearchTree
	rc uint64
	right * BinSearchTree
}

// insert x to a tree
func (t * BinSearchTree) insert(x uint64) * BinSearchTree {
	if t == nil {
		return &BinSearchTree{x, 0, nil, 0, nil}
	} else if x <= t.val {
		return &BinSearchTree{t.val, t.lc + 1, t.left.insert(x), t.rc, t.right}
	} else {
		return &BinSearchTree{t.val, t.lc, t.left, t.rc + 1, t.right.insert(x)}
	}
}

// remove the nth element in the tree (0-based), and return the removed value and the new tree
func (t * BinSearchTree) remove_nth(n uint64) (uint64, * BinSearchTree) {
	if t == nil {
		panic("remove_nth : empty tree")
	} else if n < t.lc {
		val, left_ := t.left.remove_nth(n)
		return val, &BinSearchTree{t.val, t.lc - 1, left_, t.rc, t.right}
	} else if n == t.lc {
		if t.lc < t.rc {
			val, right_ := t.right.remove_nth(0)
			return t.val, &BinSearchTree{val, t.lc, t.left, t.rc - 1, right_}
		} else if t.left != nil {
			val, left_ := t.left.remove_nth(t.lc - 1)
			return t.val, &BinSearchTree{val, t.lc - 1, left_, t.rc, t.right}
		} else {
			return t.val, nil
		}
	} else {
		val, right_ := t.right.remove_nth(n - t.lc - 1)
		return val, &BinSearchTree{t.val, t.lc, t.left, t.rc - 1, right_}
	}
}

// dump first n elements in the tree (0-based)
func (t * BinSearchTree) dump(n uint64) {
	if t != nil && n > 0 {
		t.left.dump(n)
		if t.lc < n {
			fmt.Printf("%d ", t.val)
			if t.lc + 1 < n {
				t.right.dump(n - t.lc - 1)
			}
		}
	}
}

func get_int64(args []string, i int, def_val int64) int64 {
	if i < len(args)  {
		x, _ := strconv.Atoi(args[i])
		return int64(x)
	} else {
		return def_val
	}
}

func main() {
	println("lang = go")
	m := get_int64(os.Args, 1, int64(100 * 1000))
	n := get_int64(os.Args, 2, m / 2)
	rg := RandState{123456}
	println("make a tree of m =", m, "nodes")
	var t * BinSearchTree = nil
	var _ uint64 = 0
	for i := int64(0); i < m; i++ {
		x := rg.next() % 1000000000
		//fmt.Printf("insert %d %d\n", i, x)
		t = t.insert(x)
	}
	println("insert/delete n =", n, "times")
	t0 := time_ns();
	for i := int64(0); i < n; i++ {
		x := rg.next() % 1000000000
		k := rg.next() % uint64(m + 1)
		t = t.insert(x)
		_, t = t.remove_nth(k)
	}
	t1 := time_ns()
	fmt.Print("dump the first 5 elements in the tree : ")
	t.dump(5)
	fmt.Println()
	fmt.Printf("took %d nsec to insert/remove %d elements (%f nsec/elem)\n", (t1 - t0), n, float64(t1 - t0) / float64(n))
}



In [ ]:
%%bash_

cat go/bst/bst.go

## 3-4. Build

In [ ]:
%%bash_

export PATH=${PATH}:~/.local/go/bin:~/go/bin
cd go/bst
go build

## 3-5. Run

In [ ]:
%%bash_

lang=go
exe=go/bst/bst

M=100000
N=500000

/usr/bin/time ${exe} ${M} ${N}


## 3-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=bst.md

Problem:
{problem}

...

# 4. Julia

## 4-1. AI tutor

In [ ]:
import heytutor

## 4-2. Set up a module
* Unnecessary, but make a directory for organization consistent with other languages


In [ ]:
%%bash_

mkdir -p jl/bst

## 4-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* If you edit the file outside Jupyter, <font color=red>be careful not to overwrite it with an empty file</font>

In [ ]:
%%writefile_ jl/bst/bst.jl

#!/usr/bin/env julia
# ------------- random number generator -------------

"""
random number generator state
"""
mutable struct RandState
    x :: UInt64
end

"""
next number
"""
function next(rg :: RandState)
    x = rg.x
    a = UInt64(0x5DEECE66D)
    c = UInt64(0xB)
    m = UInt64(2)^48 - 1
    rg.x = (a * x + c) & m
    rg.x >> 17
end

# ------------- binary search tree -------------

"""
binary search tree
"""
mutable struct BinSearchTree
    val :: UInt64
    lc :: UInt64
    left :: Union{BinSearchTree,Nothing}
    rc :: UInt64
    right :: Union{BinSearchTree,Nothing}
end

"""
insert x to an empty tree
"""
function insert(t :: Nothing, val :: UInt64)
    BinSearchTree(val, 0, nothing, 0, nothing)
end

"""
insert x to a non-empty tree
"""
function insert(t :: BinSearchTree, val :: UInt64)
    if val <= t.val
        BinSearchTree(t.val, t.lc + 1, insert(t.left, val), t.rc, t.right)
    else
        BinSearchTree(t.val, t.lc, t.left, t.rc + 1, insert(t.right, val))
    end
end

"""
remove the n-th smallest element from a tree
"""
function remove_nth(t :: Nothing, n :: UInt64)
    error("remove_nth : empty tree")
end

function remove_nth(t :: BinSearchTree, n :: UInt64)
    if n < t.lc
        val, left_ = remove_nth(t.left, n)
        val, BinSearchTree(t.val, t.lc - 1, left_, t.rc, t.right)
    elseif n == t.lc
        if t.lc < t.rc
            val, right_ = remove_nth(t.right, UInt64(0))
            t.val, BinSearchTree(val, t.lc, t.left, t.rc - 1, right_)
        elseif t.left != nothing
            val, left_ = remove_nth(t.left, t.lc - 1)
            t.val, BinSearchTree(val, t.lc - 1, left_, t.rc, t.right)
        else
            t.val, nothing
        end
    else
        val, right_ = remove_nth(t.right, n - t.lc - 1)
        val, BinSearchTree(t.val, t.lc, t.left, t.rc - 1, right_)
    end
end

"""
dump first n elements of the tree
"""
function dump(t :: Nothing, n :: UInt64)
    nothing
end

function dump(t :: BinSearchTree, n :: UInt64)
    if t != nothing && n > 0
        dump(t.left, n)
        if t.lc < n
            print(t.val, " ")
            if t.lc + 1 < n
                dump(t.right, n - t.lc - 1)
            end
        end
    end
end

function main()
    println("lang = julia")
    argc = length(ARGS)
    m = if argc >= 1 parse(Int64, ARGS[1]) else 100 * 1000 end
    n = if argc >= 2 parse(Int64, ARGS[2]) else div(m, 2) end
    rg = RandState(123456)
    println("make a tree of m = ", m, " nodes")
    t = nothing
    # make a tree of m nodes
    for i = 1:m
        x = next(rg) % 1000000000
        t = insert(t, x)
    end
    println("insert/delete n = ", n, " times")
    t0 = time_ns()
    # repeat n times removing an element and inserting another
    for i = 1:n
        x = next(rg) % 1000000000
        k = next(rg) % (m + 1)
        t = insert(t, x)
        v, t = remove_nth(t, k)
    end
    t1 = time_ns()
    print("dump the first 5 elements in the tree : ")
    dump(t, UInt64(5))
    println("")
    println("took ", (t1 - t0), " nsec to insert/remove ", n, " elements (", (t1 - t0) / n, " nsec/elem)")
end

main()


    


In [ ]:
%%bash_

cat jl/bst/bst.jl

## 4-4. Build
* Unnecessary, but we make the Julia file executable for consistency with other languages

In [ ]:
%%bash_

chmod +x jl/bst/bst.jl

## 4-5. Run

In [ ]:
%%bash_
export PATH=${PATH}:~/.juliaup/bin

lang=jl
exe=jl/bst/bst.jl

M=100000
N=500000

/usr/bin/time ${exe} ${M} ${N}


* Note: when you run `julia` or other Julia commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`export PATH=${PATH}:~/.juliaup/bin`)
* You may consider adding that line in your `~/.bash_profile`

## 4-6. Ask Questions or Get Feedback
* Consider including `{bash[-1]}` --- the last output by `%%bash_` --- to get feedback on errors


In [ ]:
%%hey problem_file=bst.md

Problem:
{problem}

...

# 5. OCaml

## 5-1. AI tutor

In [ ]:
import heytutor

## 5-2. Set up a module

In [ ]:
%%bash_

eval $(opam env)
mkdir -p ml
cd ml
dune init proj bst

* Note: when you run `ocamlc` or other OCaml commands (see below) in a terminal (SSH or Jupyter terminal), you need to execute the first line (`eval $(opam env)`)
* You may consider adding that line in your `~/.bash_profile`

## 5-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* `dune init proj bst` command above should have created the following file
* <font color=red>Be careful not to overwrite it with an empty file</font>

In [ ]:
%%bash_

cat ml/bst/bin/main.ml

In [ ]:
%%writefile_ ml/bst/bin/main.ml

(* get current time in ns *)
exception Couldnt_get_time
;;

let time_ns () = 
  match Base.Int63.to_int (Time_now.nanoseconds_since_unix_epoch()) with
    Some(t) -> t
  | None -> raise Couldnt_get_time
;;

(* ------------- random number generator ------------- *)

(* random number generator state *)
class rand_state x = object
  val mutable x = x
  method next () =
    let a = 0x5DEECE66D in
    let c = 0xB in
    let m = (Int.shift_left 1 48) - 1 in
    x <- (a * x + c) land m; Int.shift_right x 17
end

(* binary search tree *)
type bin_search_tree =
  Empty
| Node of (int * int * bin_search_tree * int * bin_search_tree) (* val, lc, left, rc, right *)
;;

(* insert x to a tree *)
let rec insert t x =
  match t with
    Empty -> Node(x, 0, Empty, 0, Empty)
  | Node(v, lc, left, rc, right) ->
     if x <= v then
       Node(v, lc + 1, (insert left x), rc, right)
     else
       Node(v, lc, left, rc + 1, (insert right x))
;;

(* remove n-th element *)
let rec remove_nth t n = 
  match t with
    Empty -> failwith "remove_nth: empty tree"
  | Node(v, lc, left, rc, right) ->
     if n < lc then
       let (v', left') = remove_nth left n in
       (v', Node(v, lc - 1, left', rc, right))
     else if n == lc then
       if lc < rc then
         let (v', right') = remove_nth right 0 in
         (v, Node(v', lc, left, rc - 1, right'))
       else
         match left with
           Empty -> (v, Empty)
         | _ ->
            let (v', left') = remove_nth left (lc - 1) in
            (v, Node(v', lc - 1, left', rc, right))
     else
       let (v', right') = remove_nth right (n - lc - 1) in
       (v', Node(v, lc, left, rc - 1, right'))
;;

(* dump the first n elements *)
let rec dump t n =
  match t with
    Empty -> ()
  | Node(v, lc, left, _rc, right) ->
     if n > 0 then
       let _ = dump left n in
       if lc < n then
         let _ = Printf.printf "%d " v in
         if lc + 1 < n then
           dump right (n - lc - 1)
;;

let main () =
  let _ = Printf.printf "lang = ocaml\n" in
  let argv = Sys.argv in
  let argc = Array.length argv in
  let m = if argc > 1 then int_of_string argv.(1) else 100 * 1000 in
  let n = if argc > 2 then int_of_string argv.(2) else m / 2 in
  let rg = new rand_state 123456 in
  (* insert m elements *)
  let _ = Printf.printf "make a tree of m = %d nodes\n" m in
  let rec insert_loop i m t =
    if i = m then
      t
    else
       let x = (rg#next ()) mod 1000000000 in
       let t' = insert t x in
       insert_loop (i + 1) m t'
  in
  let t = insert_loop 0 m Empty in
  (* insert an element and remove the maximum element *)
  let _ = Printf.printf "insert/delete n = %d times\n" n in
  (* let h = mk_heap n_records in *)
  let rec insert_delete_loop i n t =
    if i = n then
      t
    else
      let x = (rg#next ()) mod 1000000000 in
      let k = (rg#next ()) mod (m + 1) in
      let t' = insert t x in
      let _v, t'' = remove_nth t' k in
      insert_delete_loop (i + 1) n t''
  in
  let t0 = time_ns () in 
  (* repeat n times removing an element and inserting another *)
  let t = insert_delete_loop 0 n t in
  let t1 = time_ns () in
  let _ = Printf.printf "dump the first 5 elements in the tree : " in
  let _ = dump t 5 in
  let _ = Printf.printf "\n" in
  Printf.printf "took %d nsec to insert/remove %d elements (%f nsec/elem)\n" (t1 - t0) n ((Float.of_int (t1 - t0)) /. (Float.of_int n))
;;

main()



## 5-4. Build

In [ ]:
%%bash_

eval $(opam env)
cd ml/bst
dune build --release

## 5-5. Run

In [ ]:
%%bash_

lang=ml
exe=ml/bst/_build/default/bin/main.exe

M=100000
N=500000

/usr/bin/time ${exe} ${M} ${N}


## 5-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=bst.md

Problem:
{problem}

...

# 6. Rust

## 6-1. AI tutor

In [ ]:
import heytutor

## 6-2. Set up a module

In [ ]:
%%bash_

. ~/.cargo/env
mkdir -p rs
cd rs
cargo new bst

* Note: when you run `rustc` or other Rust commands in a terminal (SSH or Jupyter terminal), you need to execute the first line (`. ~/.cargo/env`)
* You may consider adding that line in your `~/.bash_profile`

## 6-3. Writing code
* Write it from scratch below
* No boilerplate provided
* Add `%%writefile_` cells if there are multiple files and you want to write code in Jupyter
* `cargo new bst` command above should have created the following file
* <font color=red>Be careful not to overwrite it with an empty file</font>

In [ ]:
%%bash_

cat rs/bst/src/main.rs

In [ ]:
%%writefile_ rs/bst/src/main.rs

// ---------- random number generator state ----------

struct RandState {
    x : u64
}

// next number
fn next(rg : &mut RandState) -> u64 {
    let x = rg.x;
    let a = 0x5DEECE66D as u64;
    let c = 0xB as u64;
    let m = ((1 as u64) << 48) - 1;
    rg.x = a.wrapping_mul(x).wrapping_add(c) & m;
    rg.x >> 17
}

// ---------- binary search tree ----------

enum BinSearchTree {
    Empty,
    Node{val: u64, lc: u64, left: Box<BinSearchTree>, rc: u64, right: Box<BinSearchTree>}
}

// insert x to a tree
fn insert(t : BinSearchTree, x: u64) -> BinSearchTree {
    match t {
        BinSearchTree::Empty =>
            BinSearchTree::Node{val: x,
				lc: 0,
                                left: Box::new(BinSearchTree::Empty),
				rc: 0,
                                right: Box::new(BinSearchTree::Empty)},
        BinSearchTree::Node{val, lc, left, rc, right} => 
            if x <= val {
                BinSearchTree::Node{val: val, lc: lc + 1, left: Box::new(insert(*left, x)), rc, right}
            } else {
                BinSearchTree::Node{val: val, lc, left, rc: rc + 1, right: Box::new(insert(*right, x))}
            }
    }
}

// remove the nth element in the tree (0-based), and return the removed value and the new tree
fn remove_nth(t : BinSearchTree, n : u64) -> (u64, BinSearchTree) {
    match t {
	BinSearchTree::Empty => panic!("remove_nth: empty tree"),
	BinSearchTree::Node{val, lc, left, rc, right} =>
	    if n < lc {
		let (val_, left_) = remove_nth(*left, n);
		(val_, BinSearchTree::Node{val, lc: lc - 1, left: Box::new(left_), rc, right})
	    } else if n == lc {
		if lc < rc {
		    let (val_, right_) = remove_nth(*right, 0);
		    (val, BinSearchTree::Node{val: val_, lc, left, rc: rc - 1, right: Box::new(right_)})
		} else {
		    match *left {
			BinSearchTree::Empty => (val, BinSearchTree::Empty),
			_ => {
			    let (val_, left_) = remove_nth(*left, lc - 1);
			    (val, BinSearchTree::Node{val: val_, lc: lc - 1, left: Box::new(left_), rc, right})
			}
		    }
		}
	    } else {
		let (val_, right_) = remove_nth(*right, n - lc - 1);
		(val_, BinSearchTree::Node{val, lc, left, rc: rc - 1, right: Box::new(right_)})
	    }
    }
}

// dump the first n elements in the tree
fn dump(t : BinSearchTree, n : u64) {
    match t {
	BinSearchTree::Empty => (),
	BinSearchTree::Node{val, lc, left, rc : _, right} =>
	    if n > 0 {
		dump(*left, n);
		if lc < n {
		    print!("{} ", val);
		    if lc + 1 < n {
			dump(*right, n - lc - 1);
		    }
		}
	    }
    }
}

fn main() {
    println!("lang = rust");
    let args : Vec<String> = std::env::args().collect();
    let argc = args.len();
    let m = if argc > 1 { args[1].parse::<usize>().unwrap() } else { 100_000 };
    let n = if argc > 2 { args[2].parse::<usize>().unwrap() } else { m / 2 };
    let mut rg = RandState{x: 123456};
    println!("make a tree of m = {} nodes", m);
    let mut t = BinSearchTree::Empty;
    let mut _v : u64 = 0;
    for _i in 0..m {
        let x = next(&mut rg) % 1_000_000_000;
        t = insert(t, x);
    }
    println!("insert/delete n = {} times", n);
    let t0 = std::time::Instant::now();
    for _i in 0..n {
        let x = next(&mut rg) % 1_000_000_000;
	let k = next(&mut rg) % (m + 1) as u64;
        t = insert(t, x);
        (_v, t) = remove_nth(t, k);
    }
    let dt = t0.elapsed().as_nanos();
    print!("dump the first 5 elements in the tree : ");
    dump(t, 5);
    println!();
    println!("took {} nsec to insert/remove {} elements ({} nsec/elem)", dt, n, dt as f64 / n as f64);
}



## 6-4. Build

In [ ]:
%%bash_

. ~/.cargo/env
cd rs/bst
cargo build --release

## 6-5. Run

In [ ]:
%%bash_

lang=rs
exe=rs/bst/target/release/bst

M=100000
N=500000

/usr/bin/time ${exe} ${M} ${N}


## 6-6. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=bst.md

Problem:
{problem}

...


# 7. Summarize your observations in this experiment
* Summarize your observations in this experiment below


In [ ]:
%%writefile_ note.md


# 8. Ask Questions or Get Feedback

In [ ]:
%%hey problem_file=bst.md

Problem:
{problem}

My thoughts: note.md
{file:note.md}

Give me feedback on my thoughts.